# Auditoria interna da Fase 3.9

Esta auditoria **não treina, não recalibra e não usa Mirage**. Lê cabeçalhos das imagens de treino e seleção e infere **somente a seleção interna** com os checkpoints congelados das fases 3.8 e 3.9. Confere por manifesto as divisões de calibração e do teste OpenFake reservado, sem abrir suas imagens.

Os dois modelos são comparados nas mesmas 1.420 imagens de seleção já presentes na 3.8 e nas 256 imagens TGIF adicionadas à seleção da 3.9. Cada modelo usa sua temperatura e seu limiar próprios, definidos anteriormente na calibração interna. AUC também é apresentada para separar ordenação de decisões no limiar. A seleção já foi usada para escolher checkpoints: os números são **descritivos**, não um teste independente nem uma prova causal da queda no Mirage.

Execute #1 a #5 para preparar o runtime; #6 copia os dados internos congelados para `/content`; #7 faz a auditoria; #8 lê o relatório, #9 desenha a comparação e #10 mostra uma revisão visual opcional de erros internos. Não execute o treino novamente. Em uma nova sessão, repita #1 a #6; #7 preserva um relatório já concluído.


In [ ]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

In [ ]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_9', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

In [ ]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase39-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_8_aigi_piloto.py', 'experimentos/fase3_8_aigi_dados.py', 'experimentos/fase3_9_tgif_piloto.py', 'experimentos/fase3_9_tgif_dados.py', 'experimentos/fase3_9_tgif_finetune.py', 'experimentos/avaliar_fase3_9_mirage.py', 'experimentos/fase3_5_dados.py', 'experimentos/fase3_7_dados.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc39_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.9 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_9_tgif_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_9_tgif_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

In [ ]:
# 6. COPIA LOCAL: restaura dados internos congelados para a auditoria.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
CF_DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
AIGI_DATA = CODE_ROOT / 'experimentos/fase3_8_aigi_dados.py'
for label, script in [('CF 3.5', OLD_DATA), ('CF 3.7', CF_DATA), ('AIGI 3.8', AIGI_DATA), ('TGIF 3.9', DATA)]:
    run_live('copia local '+label, [sys.executable, '-u', str(script), 'stage', '--root', str(ROOT)])


In [ ]:
# 7. AUDITORIA MANUAL: compara apenas a seleção interna; não treina nem avalia Mirage.
AUDIT = CODE_ROOT / 'experimentos/auditoria_interna_fase3_9.py'
if not AUDIT.is_file():
    raise FileNotFoundError('Publique o script de auditoria na branch e execute #4 e #5 novamente')
run_live('auditoria interna 3.9', [sys.executable, '-u', str(AUDIT),
                                  '--root', str(ROOT), '--code-root', str(CODE_ROOT)])


In [ ]:
# 8. LEITURA SOMENTE-LEITURA: resumo e segmentos internos; não infere.
import csv
report = ROOT / 'resultados/auditoria_interna_fase3_9_v1'
summary_path = report / 'summary.json'
if not summary_path.is_file():
    print('Auditoria ainda não concluída; execute #7 e confira o log.')
else:
    audit_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(audit_summary, indent=2, ensure_ascii=False))
    with (report / 'segments_selection_3_9.csv').open(newline='', encoding='utf-8') as handle:
        segments = list(csv.DictReader(handle))
    eligible = [row for row in segments if row['dimension'] in ('dataset', 'group', 'category', 'dataset_size_band')
                and row['value'] and int(row['n']) >= 10]
    print('Segmentos internos com 10+ imagens e maior taxa de erro da 3.9 (descritivo):')
    for row in sorted(eligible, key=lambda item: -float(item['error_rate']))[:20]:
        print(row['dimension'], row['value'], 'classe=', row['label'],
              'erros=', row['errors'] + '/' + row['n'])


In [ ]:
# 9. VISUALIZAÇÃO: acurácia e AUC na mesma seleção interna; sem ajuste.
import matplotlib.pyplot as plt
if not summary_path.is_file():
    raise FileNotFoundError('Execute #7 antes da visualização')
metrics = json.loads(summary_path.read_text(encoding='utf-8'))['selection_metrics_descriptive']
models = ('fase3_8_aigi_pares256_v1', 'fase3_9_tgif_sd2fr_v1')
subsets = (('common_3_8', 'Seleção comum (1.420)'), ('tgif_new', 'TGIF novo (256)'))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (key, title) in zip(axes, subsets):
    for i, model in enumerate(models):
        values = metrics[model][key]
        ax.bar(i*3, values['accuracy'], color='#4472c4',
               label='Acurácia' if i == 0 else '_nolegend_')
        ax.bar(i*3+1, values['auc'], color='#70ad47',
               label='AUC' if i == 0 else '_nolegend_')
    ax.set_xticks([0.5, 3.5], ['3.8', '3.9'])
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.2)
axes[0].set_ylabel('Proporção')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2)
fig.suptitle('Diagnóstico interno descritivo: checkpoints e conjuntos fixos', y=1.04)
plt.tight_layout()
plt.show()
print('A seleção já influenciou os checkpoints. Não use este gráfico como teste externo independente.')


In [ ]:
# 10. REVISÃO VISUAL OPCIONAL: até seis FP e seis FN da seleção interna.
from PIL import Image, ImageOps
with (report / 'errors_selection_3_9.csv').open(newline='', encoding='utf-8') as handle:
    wrong = list(csv.DictReader(handle))
fp = sorted((r for r in wrong if r['error_type'] == 'FP'),
            key=lambda r: -float(r['probability_fake_3_9']))[:6]
fn = sorted((r for r in wrong if r['error_type'] == 'FN'),
            key=lambda r: float(r['probability_fake_3_9']))[:6]
fig, axes = plt.subplots(2, 6, figsize=(15, 7))
for ax, row in zip(axes.flat, fp + fn):
    with Image.open(row['path']) as opened:
        picture = ImageOps.exif_transpose(opened).convert('RGB')
    picture.thumbnail((300, 240))
    ax.imshow(picture)
    ax.set_title(f"{row['error_type']} | p={float(row['probability_fake_3_9']):.3f}\n{row['dataset']} | {row['group'][:28]}", fontsize=8)
    ax.axis('off')
for ax in list(axes.flat)[len(fp) + len(fn):]:
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Aparência semelhante não prova mesma origem; revisar apenas como hipótese visual.')
